# 02 · BEATs-to-brain encoding pipeline

## Objective

For each of the four subjects, fit encoding models using **all 12 BEATs transformer layers** and predict the 1,000 Schaefer-parcellated fMRI time series. The data split is Season 2 Episodes 1–4 for training/cross-validation and Episode 5 as the held-out test set.

The core stages are:

1. restore the Algonauts data to local disk;
2. extract mono 16-kHz audio from the selected video clips;
3. extract and save the 12 layer-wise BEATs features per fMRI TR;
4. tune and freeze the temporal lags using training episodes only;
5. run encoding models for all configured subjects.

**Out of scope here:** hierarchy assessment, anatomical ROI analysis, brain-surface maps, and spatial visualizations. Those files are retained separately for a later step.

**Important:** do not inspect Episode 5 performance and then change lag candidates, lag settings, or the alpha grid. The held-out episode is reserved for the final evaluation.


## 1. Environment and project setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import sys

# Update this path only if the project folder is stored elsewhere in Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/BEATs_algonauts_project')
if not (PROJECT_DIR / 'config.py').exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_DIR}. Update PROJECT_DIR above.')
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

os.environ['BEATS_BRAIN_IN_COLAB'] = '1'  # inherited by tune_lags.py and run_encoding.py
os.environ['BEATS_BRAIN_DRIVE_ROOT'] = str(PROJECT_DIR)
os.environ['BEATS_BRAIN_WORK_DIR'] = str(PROJECT_DIR / 'outputs')

import config
print('Project:', PROJECT_DIR)
print('Subjects:', config.SUBJECTS)
print('Training episodes:', config.TRAIN_EPS, '| held-out episode:', config.TEST_EP)
print('Working outputs:', config.WORK_DIR)


Mounted at /content/drive
Project: /content/drive/MyDrive/BEATs_algonauts_project
Subjects: ['sub-01', 'sub-02', 'sub-03', 'sub-05']
Training episodes: [1, 2, 3, 4] | held-out episode: 5
Working outputs: /content/drive/MyDrive/BEATs_algonauts_project/outputs


In [2]:
# Colab normally provides PyTorch and torchaudio. Install only the lightweight data dependencies here.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'soundfile'], check=True)

if shutil.which('ffmpeg') is None:
    raise EnvironmentError('ffmpeg is required for audio extraction but was not found on PATH.')


## 2. Restore and verify the Algonauts inputs

In [3]:
import get_algonauts_data as algonauts

if config.DRIVE_DATA is None or not config.DRIVE_DATA.exists():
    raise FileNotFoundError(
        f'Algonauts Drive backup not found at {config.DRIVE_DATA}. '
        'Run notebook 01 first and check config.DRIVE_DATA.'
    )

copied, skipped = algonauts.restore_from_drive(config.DRIVE_DATA, config.ALGONAUTS_ROOT)
print(f'Local data restore: copied={copied}, already present={skipped}')

for subject in config.SUBJECTS:
    h5_files = list((config.FMRI_DIR / subject / 'func').glob(config.FMRI_GLOB))
    if len(h5_files) != 1:
        raise FileNotFoundError(f'{subject}: expected exactly one Friends Schaefer-1000 .h5 file, found {h5_files}')
print('fMRI inputs verified for all configured subjects.')


Local data restore: copied=14, already present=0
fMRI inputs verified for all configured subjects.


## 3. Get BEATs source code and verify the checkpoint

In [4]:
beats_repo = Path(config.BEATS_REPO)
if not (beats_repo / 'BEATs.py').exists():
    clone_root = beats_repo.parent
    clone_root.parent.mkdir(parents=True, exist_ok=True)
    if not (clone_root / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
             'https://github.com/microsoft/unilm.git', str(clone_root)],
            check=True,
        )
    subprocess.run(['git', 'sparse-checkout', 'set', 'beats'], cwd=clone_root, check=True)

checkpoint = Path(config.BEATS_CKPT)
if not checkpoint.exists():
    raise FileNotFoundError(
        f'BEATs checkpoint not found: {checkpoint}. Place BEATs_iter3_plus_AS2M.pt in the configured checkpoints folder.'
    )
print('BEATs source:', beats_repo)
print(f'Checkpoint: {checkpoint} ({checkpoint.stat().st_size / 1e6:.1f} MB)')


BEATs source: /content/unilm/beats
Checkpoint: /content/drive/MyDrive/BEATs_algonauts_project/checkpoints/BEATs_iter3_plus_AS2M.pt (361.5 MB)


## 4. Extract audio from the required clips

In [5]:
import audio_extract

audio_extract.extract_all()
print('WAV output directory:', config.WAV_DIR)

Found 10 video clips for season 2, episodes [1, 2, 3, 4, 5].
friends_s02e01a.wav:   710.1 s  (11361392 samples)  -> ~477 TRs
friends_s02e01b.wav:   710.0 s  (11360240 samples)  -> ~477 TRs
friends_s02e02a.wav:   670.8 s  (10732400 samples)  -> ~450 TRs
friends_s02e02b.wav:   670.7 s  (10730864 samples)  -> ~450 TRs
friends_s02e03a.wav:   691.6 s  (11064944 samples)  -> ~464 TRs
friends_s02e03b.wav:   691.5 s  (11063792 samples)  -> ~464 TRs
friends_s02e04a.wav:   676.1 s  (10816880 samples)  -> ~454 TRs
friends_s02e04b.wav:   676.0 s  (10815728 samples)  -> ~454 TRs
friends_s02e05a.wav:   677.7 s  (10843376 samples)  -> ~455 TRs
friends_s02e05b.wav:   677.6 s  (10842224 samples)  -> ~455 TRs
WAV output directory: /content/drive/MyDrive/BEATs_algonauts_project/outputs/wav


## 5. Extract the 12 BEATs layer representations

Each output file contains `X` with shape `(12, n_TR, 768)` and `n_tokens` with shape `(n_TR,)`. Features are independent of the fMRI lag settings, so this stage does not need to be repeated when lag parameters are tuned.

Existing feature files are skipped. The extractor processes only the configured Season 2 training and test episodes.


In [6]:
import beats_features

beats_features.extract_all(random_init=False, batch_size=8)
print('Trained BEATs features:', config.FEAT_DIR)


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


saved friends_s02e01a.npz: X(12, 477, 768), tokens/TR min=6 median=9
saved friends_s02e01b.npz: X(12, 477, 768), tokens/TR min=5 median=9
saved friends_s02e02a.npz: X(12, 451, 768), tokens/TR min=2 median=9
saved friends_s02e02b.npz: X(12, 451, 768), tokens/TR min=1 median=9
saved friends_s02e03a.npz: X(12, 465, 768), tokens/TR min=1 median=9
saved friends_s02e03b.npz: X(12, 465, 768), tokens/TR min=1 median=9
saved friends_s02e04a.npz: X(12, 454, 768), tokens/TR min=6 median=9
saved friends_s02e04b.npz: X(12, 454, 768), tokens/TR min=6 median=9
saved friends_s02e05a.npz: X(12, 455, 768), tokens/TR min=8 median=9
saved friends_s02e05b.npz: X(12, 455, 768), tokens/TR min=7 median=9
Trained BEATs features: /content/drive/MyDrive/BEATs_algonauts_project/outputs/features/beats_W10_m2


## 6. Tune and freeze the temporal lag set

Lag tuning uses only Episodes 1–4. It compares the predeclared candidate lag windows with leave-one-episode-out cross-validation. The selected lag set is saved with its provenance to `results/tuned_hparams.json`.

If no single candidate is within one standard error of the best in every subject/layer combination, the script deliberately records no consensus. In that case, inspect the per-combination results and document a decision in `PROTOCOL.md` before proceeding. Do not silently use the provisional default.


In [8]:
subprocess.run([sys.executable, str(PROJECT_DIR / 'tune_lags.py')], cwd=PROJECT_DIR, check=True)

import json
from config import TUNED_PATH
record = json.loads(Path(TUNED_PATH).read_text())
if not record.get('LAGS'):
    raise RuntimeError(
        'Lag tuning did not produce a consensus. Review results/tuned_hparams.json and document a choice before encoding.'
    )
print('Frozen LAGS:', record['LAGS'])
print('Tuning record:', TUNED_PATH)


Frozen LAGS: [1, 2, 3, 4, 5]
Tuning record: /content/drive/MyDrive/BEATs_algonauts_project/outputs/results/tuned_hparams.json


## 7. Run layer-wise encoding for all subjects

The command below fits all 12 layers for all configured subjects. For each layer and parcel, ridge alpha is selected using leave-one-training-episode-out cross-validation. The model is then refit on Episodes 1–4 and evaluated once on Episode 5.

Each subject produces a file containing:
- `cv_r`: training-only cross-validation score for every layer and parcel;
- `r_test`: held-out Episode 5 Pearson correlation for every layer and parcel;
- `best_alpha`: selected ridge regularization value per layer and parcel;
- `cv_r_folds`: fold-wise cross-validation scores at the selected alpha;
- `lags` and `lags_frozen`: protocol provenance.


In [9]:
subprocess.run([sys.executable, str(PROJECT_DIR / 'run_encoding.py')], cwd=PROJECT_DIR, check=True)

for subject in config.SUBJECTS:
    result_file = config.RESULT_DIR / f'{subject}_trained.npz'
    if not result_file.exists():
        raise FileNotFoundError(f'Expected result file was not created: {result_file}')
    with __import__('numpy').load(result_file) as z:
        print(subject, '| cv_r:', z['cv_r'].shape, '| r_test:', z['r_test'].shape,
              '| frozen lags:', z['lags'].tolist())


sub-01 | cv_r: (12, 1000) | r_test: (12, 1000) | frozen lags: [1, 2, 3, 4, 5]
sub-02 | cv_r: (12, 1000) | r_test: (12, 1000) | frozen lags: [1, 2, 3, 4, 5]
sub-03 | cv_r: (12, 1000) | r_test: (12, 1000) | frozen lags: [1, 2, 3, 4, 5]
sub-05 | cv_r: (12, 1000) | r_test: (12, 1000) | frozen lags: [1, 2, 3, 4, 5]


### 7.1 Held-out Episode 5 performance summary

The table below summarizes the **held-out test-set Pearson correlation (`r_test`)** for every BEATs layer and subject. Each value is aggregated across the 1,000 parcels. `Mean test r` and `Median test r` are descriptive summaries only; they are **not** used to select the best layer. Layer selection remains based on training-only cross-validation (`cv_r`) in notebook 03.

The complete table is also saved as `results/held_out_test_performance_summary.csv`.

In [10]:
import numpy as np
import pandas as pd
from IPython.display import display

test_summary_rows = []
for subject in config.SUBJECTS:
    result_file = config.RESULT_DIR / f"{subject}_trained.npz"
    if not result_file.exists():
        raise FileNotFoundError(f"Missing encoding results for {subject}: {result_file}")

    with np.load(result_file) as result:
        if "r_test" not in result.files:
            raise KeyError(f"{result_file} does not contain the required r_test array")
        r_test = np.asarray(result["r_test"], dtype=float)

    expected_shape = (config.N_LAYERS, config.N_PARCELS)
    if r_test.shape != expected_shape:
        raise ValueError(f"{subject}: expected r_test shape {expected_shape}, got {r_test.shape}")

    for layer_idx, parcel_scores in enumerate(r_test, start=1):
        valid_scores = parcel_scores[np.isfinite(parcel_scores)]
        test_summary_rows.append({
            "Subject": subject,
            "BEATs layer": layer_idx,
            "Mean test r": float(np.mean(valid_scores)) if valid_scores.size else np.nan,
            "Median test r": float(np.median(valid_scores)) if valid_scores.size else np.nan,
            "Valid parcels": int(valid_scores.size),
        })

test_summary = pd.DataFrame(test_summary_rows)
summary_path = config.RESULT_DIR / "held_out_test_performance_summary.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
test_summary.to_csv(summary_path, index=False)

display(test_summary.style.format({"Mean test r": "{:.4f}", "Median test r": "{:.4f}"}))
print(f"\nFull-precision summary saved to: {summary_path}")


,Subject,BEATs layer,Mean test r,Median test r,Valid parcels
0,sub-01,1,0.1619,0.1362,1000
1,sub-01,2,0.1763,0.1493,1000
2,sub-01,3,0.1814,0.1545,1000
3,sub-01,4,0.1883,0.1622,1000
4,sub-01,5,0.1906,0.1643,1000
5,sub-01,6,0.1925,0.1669,1000
6,sub-01,7,0.1950,0.1683,1000
7,sub-01,8,0.1946,0.1690,1000
8,sub-01,9,0.1932,0.1660,1000
9,sub-01,10,0.1940,0.1658,1000



Full-precision summary saved to: /content/drive/MyDrive/BEATs_algonauts_project/outputs/results/held_out_test_performance_summary.csv


## 8. Random-initialized BEATs control

The purpose is to test whether the brain-prediction performance comes from meaningful representations learned during BEATs pretraining, rather than simply from the network architecture.

It extracts a second feature set from the same BEATs architecture with random initial weights, then runs the same encoding protocol and frozen lag set. Run it only if you intend to compare trained and random-initialized representations.


In [11]:
RUN_RANDOM_INIT_CONTROL = True

if RUN_RANDOM_INIT_CONTROL:
    subprocess.run(
        [sys.executable, '-c', 'import beats_features as b; b.extract_all(random_init=True, batch_size=8)'],
        cwd=PROJECT_DIR, check=True,
    )
    subprocess.run([sys.executable, str(PROJECT_DIR / 'run_encoding.py'), '--random'], cwd=PROJECT_DIR, check=True)
    print('Random-initialized control results saved in:', config.RESULT_DIR)
else:
    print('Random-initialized control skipped.')


Random-initialized control results saved in: /content/drive/MyDrive/BEATs_algonauts_project/outputs/results


## 9. Next step

Open **`03_layerwise_results_analysis.ipynb`** to:
- identify the best layer for every parcel within each subject (selected by `cv_r`);
- identify the best layer for each parcel after averaging training-CV scores across subjects;
- identify the single best overall layer across subjects and parcels;
- inspect the corresponding Episode 5 scores as held-out confirmation.

No hierarchy inference or spatial-map visualization is performed in that notebook.
